# Derived feature creation
1. RA + diff features
2. Spatial lag

In [ ]:
import geopandas as gpd
import sqlite3
import pandas as pd
import holidays
import matplotlib.pyplot as plt
import libpysal
import seaborn as sns

conn = sqlite3.connect('../data/police_data.db')
cursor = conn.cursor()

In [ ]:
#1 RA/Diff features
#adding blank columns
cursor.execute('ALTER TABLE lsoa_month ADD COLUMN "3mo_ra" REAL;')
cursor.execute('ALTER TABLE lsoa_month ADD COLUMN "6mo_ra" REAL;')
cursor.execute('ALTER TABLE lsoa_month ADD COLUMN "12mo_diff_safe" INTEGER;')


#3 month rolling average
query_3mo = """
WITH calc AS (
    SELECT lsoa_code, month, crime_type,
           AVG(crime_count) OVER (
               PARTITION BY lsoa_code, crime_type 
               ORDER BY month 
               ROWS BETWEEN 3 PRECEDING AND 1 PRECEDING
           ) as val
    FROM lsoa_month
)
UPDATE lsoa_month
SET "3mo_ra" = calc.val
FROM calc
WHERE lsoa_month.lsoa_code = calc.lsoa_code
  AND lsoa_month.month = calc.month
  AND lsoa_month.crime_type = calc.crime_type;
"""
cursor.execute(query_3mo)

#6 month rolling average
query_6mo = """
WITH calc AS (
    SELECT lsoa_code, month, crime_type,
           AVG(crime_count) OVER (
               PARTITION BY lsoa_code, crime_type 
               ORDER BY month 
               ROWS BETWEEN 6 PRECEDING AND 1 PRECEDING
           ) as val
    FROM lsoa_month
)
UPDATE lsoa_month
SET "6mo_ra" = calc.val
FROM calc
WHERE lsoa_month.lsoa_code = calc.lsoa_code
  AND lsoa_month.month = calc.month
  AND lsoa_month.crime_type = calc.crime_type;
"""
cursor.execute(query_6mo)

#12 mo diff (t-1)-(t-13)
query_12mo = """
WITH calc AS (
    SELECT lsoa_code, month, crime_type,
           LAG(crime_count, 1) OVER (PARTITION BY lsoa_code, crime_type ORDER BY month) 
           - 
           LAG(crime_count, 13) OVER (PARTITION BY lsoa_code, crime_type ORDER BY month) as val
    FROM lsoa_month
)
UPDATE lsoa_month
SET "12mo_diff_safe" = calc.val
FROM calc
WHERE lsoa_month.lsoa_code = calc.lsoa_code
  AND lsoa_month.month = calc.month
  AND lsoa_month.crime_type = calc.crime_type;
"""
cursor.execute(query_12mo)

conn.commit()

In [ ]:
#2 Spatial lag feature

crime_df = pd.read_sql("SELECT lsoa_code, month, crime_type, crime_count FROM lsoa_month;", conn)

#get all lsoas present in our crime data
valid_lsoas = crime_df['lsoa_code'].unique()

#loading geojson + filtering out missing 35 lsoas
gdf = gpd.read_file('../data/lsoa_spatial.geojson')
gdf = gdf[gdf['LSOA21CD'].isin(valid_lsoas)]

#sorting df
gdf = gdf.sort_values('LSOA21CD').reset_index(drop=True)

#building pysal spatial weights matrix; queen contiguity considers polygons as neighbors if they share at least one edge or vertex 
w = libpysal.weights.Queen.from_dataframe(gdf, idVariable='LSOA21CD')

#row-standardize weights
w.transform = 'R'

#changed from tutorial; makes a wide table where rows = lsoa columns=month,crime_type
wide_crimes = crime_df.pivot(index='lsoa_code', columns=['month', 'crime_type'], values='crime_count')

#reordering df rows to match pysal weights matrix id order
wide_crimes = wide_crimes.reindex(w.id_order)

#calculate the spatial lag for every month and crime type at same time; standard matrix multiplication: Y_lag = W * Y
lagged_wide = pd.DataFrame(
    w.sparse.dot(wide_crimes.fillna(0)), 
    index=w.id_order, 
    columns=wide_crimes.columns
)

#format back to db
lagged_long = lagged_wide.reset_index().melt(
    id_vars='index', 
    value_name='spatial_lag'
).rename(columns={'index': 'lsoa_code'})

#back to db
cursor.execute("ALTER TABLE lsoa_month ADD COLUMN spatial_lag REAL;")
lagged_long.to_sql('temp_spatial_lag', conn, if_exists='replace', index=False)

update_query = """
UPDATE lsoa_month
SET spatial_lag = temp_spatial_lag.spatial_lag
FROM temp_spatial_lag
WHERE lsoa_month.lsoa_code = temp_spatial_lag.lsoa_code
  AND lsoa_month.month = temp_spatial_lag.month
  AND lsoa_month.crime_type = temp_spatial_lag.crime_type;
"""
cursor.execute(update_query)

cursor.execute("DROP TABLE temp_spatial_lag;")
conn.commit()


In [ ]:
#2a moran plot
conn = sqlite3.connect('../data/police_data.db')
df = pd.read_sql("""
    SELECT crime_count, spatial_lag 
    FROM lsoa_month 
    WHERE crime_type = 'Violence' 
      AND month = '2023-01'
""", conn)

plt.figure(figsize=(8, 8))
sns.regplot(
    x='crime_count', 
    y='spatial_lag', 
    data=df, 
    scatter_kws={'alpha':0.3, 'color': 'steelblue'}, 
    line_kws={'color':'red'}
)

plt.title("Moran Scatter Plot", fontsize=14)
plt.xlabel("Actual Crime Count", fontsize=12)
plt.ylabel("Spatial Lag", fontsize=12)

plt.grid(True, linestyle='--', alpha=0.6)
plt.show()

In [ ]:
#9c done w/ gemini; maps of spatial vs. actual crime
df_map = pd.read_sql("""
    SELECT lsoa_code, crime_count, spatial_lag 
    FROM lsoa_month 
    WHERE crime_type = 'Violence' 
      AND month = '2023-01'
""", conn)

gdf = gpd.read_file('../data/lsoa_spatial.geojson')
gdf_merged = gdf.merge(df_map, left_on='LSOA21CD', right_on='lsoa_code', how='inner')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 10))

gdf_merged.plot(
    column='crime_count', 
    cmap='OrRd', # Orange-Red color scale
    linewidth=0.1, 
    edgecolor='grey', 
    legend=True, 
    ax=ax1
)
ax1.set_title('RAW DATA: Actual Crime Count', fontsize=16)
ax1.axis('off')

gdf_merged.plot(
    column='spatial_lag', 
    cmap='OrRd', 
    linewidth=0.1, 
    edgecolor='grey', 
    legend=True, 
    ax=ax2
)
ax2.set_title('SPATIAL LAG: Neighbor Average', fontsize=16)
ax2.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
conn.close()